In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

import boto3
import os

session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

In [4]:
import mlflow

mlflow.set_tracking_uri("http://ec2-98-89-26-222.compute-1.amazonaws.com:5000")

In [5]:
mlflow.set_experiment('Exp 5 - ML Algos with HP Tuning')

<Experiment: artifact_location='s3://mlflow-bucket-carlos/5', creation_time=1772598387409, experiment_id='5', last_update_time=1772598387409, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}>

In [6]:
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df = pd.read_csv('reddit_preprocessing.csv')
df.dropna(inplace = True)
print(df.shape)

df.head()

(36662, 2)


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [8]:
df.category = df.category.map({-1:2, 0:0, 1:1})

df.dropna(subset = ['category'], inplace = True)

ngram_range = (1,3)
max_features = 1000

vectorizer = TfidfVectorizer(ngram_range = ngram_range, max_features = max_features)
    
X_train, X_test, y_train, y_test = train_test_split(df.clean_comment, df.category, test_size = 0.2, random_state = 42, stratify = df.category)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

smote = SMOTE(random_state = 42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag('mlflow.runName', f'{model_name}_SMOTE_TFIDF_Trigrams')
        mlflow.set_tag('experiment_type','algorithm_comparision')
        
        mlflow.log_param('algo_name', model_name)
        
        model.fit(X_train_vec, y_train)
        
        y_pred = model.predict(X_test_vec)
        
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric('accuracy', accuracy)
        
        classification_rep = classification_report(y_test, y_pred, output_dict = True)
    
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f'{label}_{metric}', value)

        
        mlflow.sklearn.log_model(
            model,
            f"{model_name}_model",
            input_example=X_train_vec[:5]
        )
        
    print(f'Accuracy: {accuracy}')
    
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log = True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    
    model = XGBClassifier(n_estimators = n_estimators, learning_rate = learning_rate, max_depth = max_depth, random_state = 42)
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))

def run_optuna_experiment():
    study = optuna.create_study(direction = 'maximize')
    study.optimize(objective_xgboost, n_trials = 30)
    
    best_params = study.best_params
    best_model = XGBClassifier(**best_params)
    
    log_mlflow('XGBoost',best_model, X_train_vec, X_test_vec, y_train, y_test)
    

run_optuna_experiment()
        

[I 2026-03-04 01:46:33,644] A new study created in memory with name: no-name-3d43dad8-cdbe-40e0-8e78-f1c2973f9767
[I 2026-03-04 01:46:41,347] Trial 0 finished with value: 0.617891722351016 and parameters: {'n_estimators': 75, 'learning_rate': 0.00503228357798668, 'max_depth': 9}. Best is trial 0 with value: 0.617891722351016.
[I 2026-03-04 01:46:45,065] Trial 1 finished with value: 0.7062593754261557 and parameters: {'n_estimators': 172, 'learning_rate': 0.051522949777973906, 'max_depth': 4}. Best is trial 1 with value: 0.7062593754261557.
[I 2026-03-04 01:46:49,310] Trial 2 finished with value: 0.5795717987181236 and parameters: {'n_estimators': 266, 'learning_rate': 0.005545489893279152, 'max_depth': 3}. Best is trial 1 with value: 0.7062593754261557.
[I 2026-03-04 01:46:51,805] Trial 3 finished with value: 0.5926633028774035 and parameters: {'n_estimators': 67, 'learning_rate': 0.0119719483452883, 'max_depth': 5}. Best is trial 1 with value: 0.7062593754261557.
[I 2026-03-04 01:47:2

🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/5/runs/3dcb4625b6134b06bb7abdfcf21e037e
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/5
Accuracy: 0.767216691667803
